<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/lab-cont-2026-2/blob/main/ambiente/lab00_introducao_python_control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Lab 00 — Ambiente Python e introdução à `python-control`

**Laboratório de Controle Automático — Engenharia Elétrica — 7º período — Ifes — campus Guarapari**

**Objetivos desta aula:**
1. Verificar o ambiente de trabalho (Python, Jupyter, `python-control`);
2. Conhecer os objetos fundamentais da biblioteca: `TransferFunction` e `StateSpace`;
3. Executar a primeira simulação e o primeiro diagrama de Bode;
4. Aprender o fluxo de trabalho que será usado em todo o curso.

**Referências:** [Documentação da python-control 0.10.2](https://python-control.readthedocs.io/en/0.10.2/index.html) ·
Åström & Murray, *Feedback Systems*, cap. 1 · Curso [CDS 110 (Caltech)](https://murray.cds.caltech.edu/CDS_110/ChE_105,_Spring_2024)

In [ ]:
# Se estiver no Google Colab, descomente a linha abaixo:
# %pip install "control>=0.10,<0.11"

# importações padronizadas do curso
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

print("Versão da python-control:", ct.__version__)

## 1. Criando uma função de transferência

O objeto central do curso é a **função de transferência** $G(s)$, criada com `ct.tf(num, den)`,
onde `num` e `den` são listas com os coeficientes dos polinômios em ordem decrescente de potência.

Exemplo: sistema de 1ª ordem $G(s) = \dfrac{2}{3s + 1}$ (ganho $K = 2$, constante de tempo $\tau = 3$ s).

In [ ]:
K = 2.0     # ganho estático
tau = 3.0   # constante de tempo [s]

G = ct.tf([K], [tau, 1])
print(G)

## 2. Inspecionando o sistema

Funções de análise básica: polos, zeros, ganho DC e amortecimento.

In [ ]:
print("Polos:", ct.poles(G))
print("Zeros:", ct.zeros(G))
print("Ganho DC:", ct.dcgain(G))

# amortecimento e frequência natural de cada polo
ct.damp(G)

## 3. Primeira simulação: resposta ao degrau

`ct.step_response` devolve um objeto com vetores de tempo e saída.
A regra dos 63,2 % deve aparecer claramente: $y(\tau) \approx 0{,}632\,K$.

In [ ]:
resp = ct.step_response(G)

plt.figure(figsize=(8, 4))
plt.plot(resp.time, resp.outputs, lw=2, label='y(t)')
plt.axhline(K, color='gray', ls='--', label='valor final K')
plt.axvline(tau, color='r', ls=':', label=r'$t = \tau$')
plt.axhline(0.632 * K, color='r', ls=':')
plt.xlabel('Tempo [s]')
plt.ylabel('Saída')
plt.title('Resposta ao degrau unitário — sistema de 1ª ordem')
plt.legend()
plt.grid(True)
plt.show()

## 4. Métricas automáticas com `step_info`

In [ ]:
info = ct.step_info(G)
for chave, valor in info.items():
    print(f"{chave:>14s}: {valor:.4g}")

## 5. Primeiro diagrama de Bode

A resposta em frequência será o tema do Lab 03; aqui apenas geramos o gráfico.

In [ ]:
plt.figure(figsize=(8, 6))
ct.bode_plot(G, dB=True)
plt.show()

## 6. Espaço de estados e conversões

O mesmo sistema pode ser representado em **espaço de estados** com `ct.ss(A, B, C, D)`.
Conversões: `ct.tf2ss` e `ct.ss2tf` (ou simplesmente `ct.tf(sys)` / `ct.ss(sys)`).

In [ ]:
# 1ª ordem: dx/dt = -(1/tau) x + (K/tau) u ; y = x
A = [[-1 / tau]]
B = [[K / tau]]
C = [[1.0]]
D = [[0.0]]

sys_ss = ct.ss(A, B, C, D)
print(sys_ss)

# conversão de volta para função de transferência (deve reproduzir G)
print(ct.tf(sys_ss))

## 7. Álgebra de blocos

Operadores aritméticos combinam sistemas: série (`*`), paralelo (`+`) e realimentação (`ct.feedback`).

In [ ]:
C1 = ct.tf([5], [1])          # controlador proporcional Kp = 5
L = C1 * G                    # malha aberta
T = ct.feedback(L, 1)         # malha fechada com realimentação unitária

print("Malha fechada T(s):", T)
print("Polo de malha fechada:", ct.poles(T))

resp_mf = ct.step_response(T)
plt.figure(figsize=(8, 4))
plt.plot(resp.time, resp.outputs, label='malha aberta (G)')
plt.plot(resp_mf.time, resp_mf.outputs, label='malha fechada (Kp = 5)')
plt.xlabel('Tempo [s]')
plt.ylabel('Saída')
plt.title('Efeito da realimentação proporcional')
plt.legend()
plt.grid(True)
plt.show()

Observe: a realimentação **acelerou** o sistema (polo mais à esquerda), mas o valor final
não é mais 1 — há **erro de regime permanente**. Esse compromisso será estudado nas
Unidades II e IV.

---
## Exercícios (entregar o notebook executado)

**E1.** Crie $G_2(s) = \dfrac{10}{s^2 + 2s + 10}$, imprima seus polos e classifique o
amortecimento com `ct.damp`. O sistema é estável?

**E2.** Trace a resposta ao degrau de $G_2$ e meça o sobressinal com `ct.step_info`.

**E3.** Feche a malha de $G_2$ com $K_p = 1, 5, 20$ e compare as três respostas ao degrau
em um único gráfico. O que acontece com a velocidade, o sobressinal e o erro de regime?

**E4.** Converta $G_2$ para espaço de estados e verifique que os autovalores da matriz $A$
(`np.linalg.eigvals`) coincidem com os polos.

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui